In [1]:
#imports
import pandas as pd
import datetime as dt
import numpy as np
import psycopg2
from sqlalchemy import create_engine,text

In [2]:
#DataFrames Created
#table_creation_date --> Dates of tables created last row from files
#coverage_cat --> Coverage_Category & Coverage Category Description
#Coverage_Grp --> Code, name and Group of coverage.

In [2]:
#Reading files
coverage = pd.read_excel('coverage_data.xlsx',sheet_name='Data')
incidence_rate = pd.read_excel('incidence_rate_data.xlsx',sheet_name='Data')
reported_cases = pd.read_excel('reported_cases_data.xlsx',sheet_name='Data')
vaccine_introduction = pd.read_excel('vaccine_introduction_data.xlsx',sheet_name='Data')
vaccine_schedule = pd.read_excel('vaccine_schedule_data.xlsx',sheet_name='Data')

In [3]:
#A dataframe of table names and the date of creation of the tables
dfs = {'Coverage':coverage,'Incident_rate':incidence_rate,'reported_cases':reported_cases,'vaccine_introduction':vaccine_introduction ,'vaccine_schedule':vaccine_schedule}
table_creation_date = pd.DataFrame({'Table_Name':['Coverage','Incident_rate','reported_cases','vaccine_introduction','vaccine_schedule']})
for key,table in dfs.items():
    if table.iloc[-1,1] is np.nan:
        table_creation_date['Created'] = str(table.iloc[-1,0]).replace('Created:','')
        table = table.drop(index=max(table.index),inplace=True)

In [4]:
for key,df in dfs.items():
    df['YEAR']=df['YEAR'].astype(int)

In [6]:
#Coverage_Category and Coverage_Category_Description
coverage_cat = coverage[['COVERAGE_CATEGORY','COVERAGE_CATEGORY_DESCRIPTION']].drop_duplicates().reset_index(drop=True)
coverage = coverage.drop('COVERAGE_CATEGORY_DESCRIPTION',axis=1)

In [7]:
region = coverage[['GROUP','CODE','NAME',]].drop_duplicates().reset_index(drop=True)
coverage = coverage.drop(['GROUP','NAME'],axis=1)


In [6]:
cols = list(vaccine_schedule.columns)
x = cols.index('TARGETPOP')

In [7]:
for index,values in vaccine_schedule.iterrows():
    if values['TARGETPOP'] is np.nan and values['TARGETPOP_DESCRIPTION']=='General/routine':
        vaccine_schedule.iloc[index,x]='GEN'

In [8]:
vaccine_schedule['SOURCECOMMENT']=['Nil' if x is np.nan else x for x in vaccine_schedule['SOURCECOMMENT']]

In [11]:
cols = list(coverage.columns)
x = cols.index('COVERAGE')

In [12]:
for index,values in coverage.iterrows():
    if (not np.isnan(values['COVERAGE'])) and (values['COVERAGE']==coverage.iloc[index-1,x]):
        if np.isnan(coverage.iloc[index,x-1]):
            coverage.iloc[index,x-1]=coverage.iloc[index-1,x-1]
        if np.isnan(coverage.iloc[index,x-2]):
            coverage.iloc[index,x-2]=coverage.iloc[index-1,x-2]
    elif np.isnan(values['COVERAGE']):
        if (not np.isnan(values['DOSES'])) and (not np.isnan(values['TARGET_NUMBER'])):
            if values['TARGET_NUMBER'] != 0:
                coverage.iloc[index,x]= round((values['DOSES']/values['TARGET_NUMBER'])*100,2)
            else:
                coverage.iloc[index,x] = 0
        elif (np.isnan(values['DOSES'])) and (not np.isnan(values['TARGET_NUMBER'])):
            coverage.iloc[index,x] = 0
            coverage.iloc[index,x-1] = 0
        elif (not np.isnan(values['DOSES'])) and (np.isnan(values['TARGET_NUMBER'])):
            coverage.iloc[index,x-2] = 0
            coverage.iloc[index,x] = 100
        else:
            coverage.iloc[index,x]=0
            coverage.iloc[index,x-1]=0
            coverage.iloc[index,x-2]=0
    if (not np.isnan(values['COVERAGE'])) and (values['COVERAGE']!=coverage.iloc[index-1,x]):
        if np.isnan(coverage.iloc[index,x-2]):
            coverage.iloc[index,x-2]=coverage.iloc[index-1,x-2]
        if np.isnan(coverage.iloc[index,x-1]):
            coverage.iloc[index,x-1]=(values['COVERAGE']/100)*coverage.iloc[index,x-2]


In [13]:
coverage['TARGET_NUMBER'] = coverage['TARGET_NUMBER'].astype(int)
coverage['DOSES'] = coverage['DOSES'].astype(int)

In [14]:
coverage.isnull().sum()

CODE                   0
YEAR                   0
ANTIGEN                0
ANTIGEN_DESCRIPTION    0
COVERAGE_CATEGORY      0
TARGET_NUMBER          0
DOSES                  0
COVERAGE               0
dtype: int64

In [15]:
incidence_rate = incidence_rate.fillna(0)
incidence_rate.isnull().sum()

GROUP                  0
CODE                   0
NAME                   0
YEAR                   0
DISEASE                0
DISEASE_DESCRIPTION    0
DENOMINATOR            0
INCIDENCE_RATE         0
dtype: int64

In [16]:
def denom(text):
    x = text.split()
    x = int(x[1].replace(',',''))
    return x

In [17]:
incidence_rate['FOR_EVERY'] = incidence_rate['DENOMINATOR'].apply(denom)

In [18]:
incidence_rate['INCIDENCE_RATE']=round((incidence_rate['INCIDENCE_RATE']/incidence_rate['FOR_EVERY'])*100,4)

In [19]:
incidence_rate.head()

,GROUP,CODE,NAME,YEAR,DISEASE,DISEASE_DESCRIPTION,DENOMINATOR,INCIDENCE_RATE,FOR_EVERY
0,COUNTRIES,ABW,Aruba,2023,CRS,Congenital rubella syndrome,"per 10,000 live births",0.0000,10000
1,COUNTRIES,ABW,Aruba,2023,DIPHTHERIA,Diphtheria,"per 1,000,000 total population",0.0000,1000000
2,COUNTRIES,ABW,Aruba,2023,INVASIVE_MENING,Invasive meningococcal disease,"per 1,000,000 total population",0.0009,1000000
3,COUNTRIES,ABW,Aruba,2023,MEASLES,Measles,"per 1,000,000 total population",0.0000,1000000
4,COUNTRIES,ABW,Aruba,2023,MUMPS,Mumps,"per 1,000,000 total population",0.0000,1000000


In [20]:
reported_cases.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84869 entries, 0 to 84868
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   GROUP                84869 non-null  object 
 1   CODE                 84869 non-null  object 
 2   NAME                 84869 non-null  object 
 3   YEAR                 84869 non-null  int32  
 4   DISEASE              84869 non-null  object 
 5   DISEASE_DESCRIPTION  84869 non-null  object 
 6   CASES                65470 non-null  float64
dtypes: float64(1), int32(1), object(5)
memory usage: 4.2+ MB


In [21]:
reported_cases['CASES'] = reported_cases['CASES'].fillna(0)

In [14]:
print('Coverage:',coverage.duplicated().sum())
print('Incident_Rate:',incidence_rate.duplicated().sum())
print('Reported_cases:',reported_cases.duplicated().sum())
print('Vaccine_Intro:',vaccine_introduction.duplicated().sum())
print('Vaccine_schedule:',vaccine_schedule.duplicated().sum())


Coverage: 0
Incident_Rate: 0
Reported_cases: 0
Vaccine_Intro: 0
Vaccine_schedule: 0


In [ ]:
vaccine_introduction['INTRODUCED'] = [1 if 'yes' in x.lower().split() or 'risk' in x.lower().split() else 0 for x in vaccine_introduction['INTRO']]

,ISO_3_CODE,COUNTRYNAME,WHO_REGION,YEAR,DESCRIPTION,INTRO,INTRODUCED
0,AFG,Afghanistan,EMRO,2023,aP (acellular pertussis) vaccine,No,0
1,AFG,Afghanistan,EMRO,2023,Hepatitis A vaccine,No,0
2,AFG,Afghanistan,EMRO,2023,Hepatitis B vaccine,Yes,1
3,AFG,Afghanistan,EMRO,2023,HepB birth dose,Yes,1
4,AFG,Afghanistan,EMRO,2023,Hib (Haemophilus influenzae type B) vaccine,Yes,1


In [19]:
def intro_type(x):
    x = x.split()
    if 'Yes' in x:
        if '(R)' in x:
            return 'R'
        if '(P)' in x:
            return 'P'
        if '(A)' in x:
            return 'A'
        if '(O)' in x:
            return 'O'
        if '(D)' in x:
            return 'D'
        else:
            return 'GEN'
    elif 'risk' in x:
        return 'HIGH RISK'
    else:
        if 'ND' in x:
            return 'ND'
        else:
            return 'GEN'
vaccine_introduction['TYPE'] = vaccine_introduction['INTRO'].apply(intro_type)

In [23]:
vaccine_introduction = vaccine_introduction.drop(columns='INTRO')

In [23]:
incidence_grps = incidence_rate[['GROUP','CODE','NAME']].drop_duplicates().reset_index(drop=True)
region = pd.merge(region,incidence_grps,'outer')
incidence_rate = incidence_rate.drop(columns=['GROUP','NAME'])
rc_grps = reported_cases[['GROUP','CODE','NAME']].drop_duplicates().reset_index(drop=True)
region = pd.merge(region,rc_grps,'outer')
reported_cases = reported_cases.drop(columns=['GROUP','NAME'])

In [24]:
vaccine_introduction['WHO_REGION'] = vaccine_introduction['WHO_REGION'].str.replace('O','')

In [24]:
vaccine_introduction.head(5)

,ISO_3_CODE,COUNTRYNAME,WHO_REGION,YEAR,DESCRIPTION,INTRODUCED,TYPE
0,AFG,Afghanistan,EMRO,2023,aP (acellular pertussis) vaccine,0,GEN
1,AFG,Afghanistan,EMRO,2023,Hepatitis A vaccine,0,GEN
2,AFG,Afghanistan,EMRO,2023,Hepatitis B vaccine,1,GEN
3,AFG,Afghanistan,EMRO,2023,HepB birth dose,1,GEN
4,AFG,Afghanistan,EMRO,2023,Hib (Haemophilus influenzae type B) vaccine,1,GEN


In [52]:
vaccine_schedule.isnull().sum()

id                       0
code                     0
year                     0
vaccinecode              0
vaccine_description      0
schedulerounds           0
targetpop                0
targetpop_description    0
geoarea                  0
sourcecomment            0
age                      0
dtype: int64

In [23]:
vaccine_schedule['GEOAREA'] = vaccine_schedule['GEOAREA'].fillna('NATIONAL')

In [9]:
vaccine_schedule['WHO_REGION'] = vaccine_schedule['WHO_REGION'].str.replace('O','')

In [10]:
vaccine_schedule.head()

,ISO_3_CODE,COUNTRYNAME,WHO_REGION,YEAR,VACCINECODE,VACCINE_DESCRIPTION,SCHEDULEROUNDS,TARGETPOP,TARGETPOP_DESCRIPTION,GEOAREA,AGEADMINISTERED,SOURCECOMMENT
0,ABW,Aruba,AMR,2023,DTAPHIBIPV,DTaP-Hib-IPV (acellular) vaccine,1.0,GEN,General/routine,NATIONAL,M2,Nil
1,ABW,Aruba,AMR,2023,DTAPHIBIPV,DTaP-Hib-IPV (acellular) vaccine,2.0,GEN,General/routine,NATIONAL,M4,Nil
2,ABW,Aruba,AMR,2023,DTAPHIBIPV,DTaP-Hib-IPV (acellular) vaccine,3.0,GEN,General/routine,NATIONAL,M6,Nil
3,ABW,Aruba,AMR,2023,DTAPHIBIPV,DTaP-Hib-IPV (acellular) vaccine,4.0,B_2YL_W,General/routine,NATIONAL,M15,Nil
4,ABW,Aruba,AMR,2023,DTAPIPV,DTaP-IPV (acellular) vaccine,5.0,B_CHILD_W,General/routine,NATIONAL,Y4,Nil


In [43]:
def age_cat(x):
    if '-' in x:
        x,y = x.split('-')
    x = list(x)
    if 'M' in x:
        x = [i for i in x if i.isdigit()]
        try:
            months = int(''.join(x))
        except:
            print(x)
            raise ValueError
        years = months/12
    elif 'W' in x:
        x = [i for i in x if i.isdigit()]
        try:
            weeks = int(''.join(x))
        except:
            print(x)
            raise ValueError
        years = weeks/52
    elif 'D' in x:
        x = [i for i in x if i.isdigit()]
        try:
            days = int(''.join(x))
        except:
            print(x)
            raise ValueError
        years = days/365
    elif 'Y' in x:
        x = [i for i in x if i.isdigit()]
        if len(x) == 0:
            return 'ANY'
        else:
            try:
                years = int(''.join(x))
            except:
                print(x)
                raise ValueError
    elif 'B' in x:
        years = 0
    else:
        return 'ANY'
    if years<1:
        return '<1'
    elif years>=1 and years<=4:
        return '1-4'
    elif years>4 and years<=12:
        return '4-12'
    elif years>12 and years<=17:
        return '13-17'
    elif years>17 and years<=39:
        return '17-39'
    elif years>39 and years <= 59:
        return '39-59'
    else:
        return '60+'
    

In [ ]:
vaccine_schedule['AGE'] = vaccine_schedule['AGEADMINISTERED'].apply(age_cat)

In [46]:
vaccine_schedule = vaccine_schedule.rename(columns={'ISO_3_CODE':'CODE','COUNTRYNAME':'NAME'})


In [32]:
vaccine_introduction = vaccine_introduction.rename(columns={'ISO_3_CODE':'CODE','COUNTRYNAME':'NAME'})

In [30]:
vaccine_schedule_reg = vaccine_schedule[['CODE','NAME','WHO_REGION']].drop_duplicates().reset_index(drop=True)
vaccine_intro_reg = vaccine_introduction[['CODE','NAME','WHO_REGION']].drop_duplicates().reset_index(drop=True)

In [ ]:
vaccine_schedule = vaccine_schedule.drop(columns=['NAME','WHO_REGION','AGEADMINISTERED'])
vaccine_introduction = vaccine_introduction.drop(columns=['NAME','WHO_REGION'])

KeyError: "['NAME', 'WHO_REGION', 'AGEADMINISTERED'] not found in axis"

In [32]:
region = pd.merge(region,vaccine_intro_reg,'left')
region = pd.merge(region,vaccine_schedule_reg,'left')
region.head()

,GROUP,CODE,NAME,WHO_REGION
0,COUNTRIES,ABW,Aruba,NaN
1,COUNTRIES,AFG,Afghanistan,EMR
2,COUNTRIES,AGO,Angola,AFR
3,COUNTRIES,AIA,Anguilla,NaN
4,COUNTRIES,ALB,Albania,EUR


In [33]:
region['GROUP'].unique()

array(['COUNTRIES', 'DEVELOPMENT_STATUS', 'GAVI_PHASE5', 'GLOBAL',
       'UNICEF_REGIONS', 'WB_LONG', 'WB_SHORT', 'WHO_REGIONS'],
      dtype=object)

In [34]:
region = region.fillna('NA')

In [35]:
region.head()

,GROUP,CODE,NAME,WHO_REGION
0,COUNTRIES,ABW,Aruba,NA
1,COUNTRIES,AFG,Afghanistan,EMR
2,COUNTRIES,AGO,Angola,AFR
3,COUNTRIES,AIA,Anguilla,NA
4,COUNTRIES,ALB,Albania,EUR


In [49]:
host = 'localhost'
database = 'Vaccination'
user = 'postgres'
pwd = 1234
port = 5432

engine = create_engine(f'postgresql+psycopg2://{user}:{pwd}@{host}:{port}/{database}')

In [37]:
coverage.insert(0,'id',range(1,len(coverage)+1))
coverage.columns = coverage.columns.str.lower()

In [53]:
for col in ['target_number','doses','coverage']:
    coverage = coverage[coverage[col]>=0]

In [54]:
coverage.head()

,id,code,year,antigen,antigen_description,coverage_category,target_number,doses,coverage
0,1,ABW,2023,BCG,BCG,ADMIN,0,0,0.00
1,2,ABW,2023,BCG,BCG,OFFICIAL,0,0,0.00
2,3,ABW,2023,DIPHCV4,"Diphtheria-containing vaccine, 4th dose (1st b...",ADMIN,1044,945,90.52
3,4,ABW,2023,DIPHCV4,"Diphtheria-containing vaccine, 4th dose (1st b...",OFFICIAL,1044,945,90.52
4,5,ABW,2023,DIPHCV5,"Diphtheria-containing vaccine, 5th dose (2nd b...",ADMIN,1219,1008,82.69


In [55]:
with engine.begin() as con:
    con.execute(text('''DROP TABLE IF EXISTS coverage;
                        CREATE TABLE coverage (
                            id INTEGER PRIMARY KEY,
                            code TEXT,
                            year INTEGER,
                            antigen TEXT,
                            antigen_description TEXT,
                            coverage_category TEXT,
                            target_number INTEGER,
                            doses INTEGER,
                            coverage REAL
                        );
                    '''))

In [56]:
coverage.to_sql('coverage',engine,if_exists='append',index=False,method='multi',chunksize=5000)

399843

In [41]:
incidence_rate.head()

,CODE,YEAR,DISEASE,DISEASE_DESCRIPTION,DENOMINATOR,INCIDENCE_RATE,FOR_EVERY
0,ABW,2023,CRS,Congenital rubella syndrome,"per 10,000 live births",0.0000,10000
1,ABW,2023,DIPHTHERIA,Diphtheria,"per 1,000,000 total population",0.0000,1000000
2,ABW,2023,INVASIVE_MENING,Invasive meningococcal disease,"per 1,000,000 total population",0.0009,1000000
3,ABW,2023,MEASLES,Measles,"per 1,000,000 total population",0.0000,1000000
4,ABW,2023,MUMPS,Mumps,"per 1,000,000 total population",0.0000,1000000


In [42]:
incidence_rate = incidence_rate.drop(columns=['DENOMINATOR','FOR_EVERY'],axis=1)
incidence_rate.insert(0,'id',range(1,len(incidence_rate)+1))
incidence_rate.columns = incidence_rate.columns.str.lower()

In [43]:
with engine.begin() as con:
    con.execute(text('''DROP TABLE IF EXISTS incidence_rate;
                        CREATE TABLE incidence_rate (
                            id INTEGER PRIMARY KEY,
                            code TEXT,
                            year INTEGER,
                            disease TEXT,
                            disease_description TEXT,
                            incidence_rate REAL
                        );
                    '''))
incidence_rate.to_sql('incidence_rate',engine,if_exists='append',index=False,method='multi',chunksize=5000)

84945

In [44]:
reported_cases.insert(0,'id',range(1,len(reported_cases)+1))
reported_cases.columns = reported_cases.columns.str.lower()

In [45]:
reported_cases.head()

,id,code,year,disease,disease_description,cases
0,1,ABW,2023,CRS,Congenital rubella syndrome,0.0
1,2,ABW,2023,DIPHTHERIA,Diphtheria,0.0
2,3,ABW,2023,INVASIVE_MENING,Invasive meningococcal disease,1.0
3,4,ABW,2023,MEASLES,Measles,0.0
4,5,ABW,2023,MUMPS,Mumps,0.0


In [46]:
with engine.begin() as con:
    con.execute(text('''DROP TABLE IF EXISTS reported_cases;
                        CREATE TABLE reported_cases (
                            id INTEGER PRIMARY KEY,
                            code TEXT,
                            year INTEGER,
                            disease TEXT,
                            disease_description TEXT,
                            cases REAL
                        );
                    '''))
reported_cases.to_sql('reported_cases',engine,if_exists='append',index=False,method='multi',chunksize=5000)

84869

In [35]:
vaccine_introduction.insert(0,'id',range(1,len(vaccine_introduction)+1))
vaccine_introduction.columns = vaccine_introduction.columns.str.lower()
vaccine_introduction.head()

,id,id,code,year,description,introduced,type
0,1,1,AFG,2023,aP (acellular pertussis) vaccine,0,GEN
1,2,2,AFG,2023,Hepatitis A vaccine,0,GEN
2,3,3,AFG,2023,Hepatitis B vaccine,1,GEN
3,4,4,AFG,2023,HepB birth dose,1,GEN
4,5,5,AFG,2023,Hib (Haemophilus influenzae type B) vaccine,1,GEN


In [43]:
with engine.begin() as con:
    con.execute(text('''DROP TABLE IF EXISTS vaccine_introduction;
                        CREATE TABLE vaccine_introduction (
                            id INTEGER PRIMARY KEY,
                            code TEXT,
                            year INTEGER,
                            description TEXT,
                            introduced INTEGER,
                            type TEXT
                        );
                    '''))
vaccine_introduction.to_sql('vaccine_introduction',engine,if_exists='append',index=False,method='multi',chunksize=5000)

138320

In [50]:
vaccine_schedule.insert(0,'id',range(1,len(vaccine_schedule)+1))
vaccine_schedule.columns = vaccine_schedule.columns.str.lower()
vaccine_schedule.head()

,id,code,year,vaccinecode,vaccine_description,schedulerounds,targetpop,targetpop_description,geoarea,sourcecomment,age
0,1,ABW,2023,DTAPHIBIPV,DTaP-Hib-IPV (acellular) vaccine,1.0,GEN,General/routine,NATIONAL,Nil,<1
1,2,ABW,2023,DTAPHIBIPV,DTaP-Hib-IPV (acellular) vaccine,2.0,GEN,General/routine,NATIONAL,Nil,<1
2,3,ABW,2023,DTAPHIBIPV,DTaP-Hib-IPV (acellular) vaccine,3.0,GEN,General/routine,NATIONAL,Nil,<1
3,4,ABW,2023,DTAPHIBIPV,DTaP-Hib-IPV (acellular) vaccine,4.0,B_2YL_W,General/routine,NATIONAL,Nil,1-4
4,5,ABW,2023,DTAPIPV,DTaP-IPV (acellular) vaccine,5.0,B_CHILD_W,General/routine,NATIONAL,Nil,1-4


In [51]:
with engine.begin() as con:
    con.execute(text('''DROP TABLE IF EXISTS vaccine_schedule;
                        CREATE TABLE vaccine_schedule (
                            id INTEGER PRIMARY KEY,
                            code TEXT,
                            year INTEGER,
                            vaccinecode TEXT,
                            vaccine_description TEXT,
                            schedulerounds INTEGER,
                            targetpop TEXT,
                            targetpop_description TEXT,
                            geoarea TEXT,
                            age TEXT,
                            sourcecomment TEXT
                        );
                    '''))
vaccine_schedule.to_sql('vaccine_schedule',engine,if_exists='append',index=False,method='multi',chunksize=5000)

8052

In [51]:
region.columns = region.columns.str.lower()
region.head()

,group,code,name,who_region
0,COUNTRIES,ABW,Aruba,NA
1,COUNTRIES,AFG,Afghanistan,EMR
2,COUNTRIES,AGO,Angola,AFR
3,COUNTRIES,AIA,Anguilla,NA
4,COUNTRIES,ALB,Albania,EUR


In [52]:
with engine.begin() as con:
    con.execute(text('''DROP TABLE IF EXISTS region;
                        CREATE TABLE region (
                            "group" TEXT,
                            code TEXT,
                            name TEXT,
                            who_region TEXT
                        );
                    '''))
region.to_sql('region',engine,if_exists='append',index=False,method='multi')

245